# Serving Qwen3.5-9B Efficiently on GPU with vLLM

In this notebook, you'll learn how to:
1. Launch a **CUDA-backed vLLM server** with the newer Qwen3.5-9B model
2. Query it using the OpenAI-compatible API
3. **Explore model behavior** with logprobs and sampling parameters
4. Observe **continuous batching** and **GPU cache** usage via live Prometheus metrics
5. Measure **prefix-cache hits** for repeated prompts

## From Quantization to Serving

In an earlier lesson, you used `llm-compressor` to quantize a model's weights from 16-bit to 4-bit using GPTQ. Now you'll **serve** a model with vLLM and interact with it through its OpenAI-compatible API.

[vLLM](https://github.com/vllm-project/vllm) is an open-source LLM inference engine that integrates key serving optimizations:

| Feature | Benefit |
|:--|:--|
| **Continuous batching** | Schedules at the token level: no wasted compute waiting for the longest request |
| **PagedAttention** | Manages KV cache in fixed-size blocks: near-zero memory waste |
| **Prefix caching** | Reuses KV cache for shared prompt prefixes across requests |
| **Quantization support** | Natively serves GPTQ, AWQ, and compressed-tensors models |
| **OpenAI-compatible API** | Drop-in replacement for applications already using the OpenAI client |

## Step 1: Start Your vLLM Server

This lesson targets **Qwen3.5-9B on a CUDA GPU**. Qwen3.5-9B is a newer February 2026 model with 9B language-model parameters, a hybrid linear/full-attention architecture, and optional vision support. The server runs in text-only mode here so GPU memory is focused on language inference. Start vLLM in a terminal before running the notebook cells.

**GPU server startup**

The first Python cell automatically starts Qwen3.5-9B with the repository's local `vllm/vllm-openai:latest` Docker image when no server is listening. It mounts your Hugging Face cache so the model is downloaded only once.

The equivalent command is:

```bash
docker run -d --name l6-vllm --gpus all --ipc host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  --entrypoint '' -p 8000:8000 \
  -v "$HOME/.cache/huggingface:/root/.cache/huggingface" \
  vllm/vllm-openai:latest vllm serve Qwen/Qwen3.5-9B \
  --served-model-name qwen3.5-9b \
  --language-model-only --reasoning-parser qwen3 \
  --dtype bfloat16 --max-model-len 8192 \
  --gpu-memory-utilization 0.65 --enable-prefix-caching
```

The `0.65` memory fraction leaves room for other GPU processes in this shared lab environment. Override it with `VLLM_GPU_MEMORY_UTILIZATION` when the GPU is otherwise idle.

Set `AUTO_START_VLLM=0` before running the notebook when you want to use an existing local or remote endpoint. Set `VLLM_URL` and `MODEL_SOURCE` to override the endpoint and model.

For a lower-memory comparison using the checkpoint from the compression lab:

```bash
MODEL_SOURCE=../models/Qwen3-8B-W4A16
```

The server exposes an OpenAI-compatible API, including `/v1/models`, `/v1/chat/completions`, and `/v1/completions`.

Run the next cell to verify CUDA and connect to the server. If the endpoint is absent, the cell starts the GPU container and waits for model loading to finish. The initial Qwen3.5 download can take several minutes.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json, math, os, shutil, subprocess, sys, time
from pathlib import Path

import requests
import torch

VLLM_URL = os.getenv("VLLM_URL", "http://localhost:8000").rstrip("/")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "Qwen/Qwen3.5-9B")
VLLM_IMAGE = os.getenv("VLLM_IMAGE", "vllm/vllm-openai:latest")
VLLM_CONTAINER = os.getenv("VLLM_CONTAINER", "l6-vllm")
GPU_MEMORY_UTILIZATION = os.getenv("VLLM_GPU_MEMORY_UTILIZATION", "0.65")
AUTO_START_VLLM = os.getenv("AUTO_START_VLLM", "1") == "1"
STARTUP_TIMEOUT = int(os.getenv("VLLM_STARTUP_TIMEOUT", "1200"))
os.makedirs("outputs", exist_ok=True)

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"Notebook CUDA: {properties.name} ({total_bytes / 1024**3:.1f} GiB total)")
    print(f"GPU memory currently free: {free_bytes / 1024**3:.1f} GiB")
else:
    print("Notebook CUDA is unavailable; the vLLM Docker container may still use a GPU.")

def server_model():
    try:
        response = requests.get(f"{VLLM_URL}/v1/models", timeout=3)
        response.raise_for_status()
        return response.json()["data"][0]["id"]
    except (requests.RequestException, KeyError, IndexError):
        return None

MODEL = server_model()
if MODEL is None and AUTO_START_VLLM and VLLM_URL == "http://localhost:8000":
    if shutil.which("docker") is None:
        raise RuntimeError("Docker is required to auto-start vLLM.")

    image_check = subprocess.run(
        ["docker", "image", "inspect", VLLM_IMAGE],
        capture_output=True, text=True,
    )
    if image_check.returncode != 0:
        raise RuntimeError(
            f"Docker image {VLLM_IMAGE!r} is unavailable. Pull it before rerunning this cell."
        )

    hf_cache = Path.home() / ".cache" / "huggingface"
    hf_cache.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["docker", "rm", "-f", VLLM_CONTAINER],
        capture_output=True, text=True,
    )

    command = [
        "docker", "run", "-d", "--name", VLLM_CONTAINER,
        "--gpus", "all", "--ipc", "host",
        "--ulimit", "memlock=-1", "--ulimit", "stack=67108864",
        "--entrypoint", "", "-p", "8000:8000",
        "-v", f"{hf_cache}:/root/.cache/huggingface",
        VLLM_IMAGE, "vllm", "serve", MODEL_SOURCE,
        "--served-model-name", "qwen3.5-9b",
        "--language-model-only", "--reasoning-parser", "qwen3",
        "--dtype", "bfloat16", "--max-model-len", "8192",
        "--gpu-memory-utilization", GPU_MEMORY_UTILIZATION,
        "--enable-prefix-caching",
    ]
    started = subprocess.run(command, capture_output=True, text=True)
    if started.returncode != 0:
        raise RuntimeError(f"Could not start vLLM:\n{started.stderr.strip()}")
    print(
        f"Started GPU container {VLLM_CONTAINER} with memory utilization "
        f"{GPU_MEMORY_UTILIZATION}; waiting for model load..."
    )

deadline = time.monotonic() + STARTUP_TIMEOUT
last_status = 0
while MODEL is None and time.monotonic() < deadline:
    if AUTO_START_VLLM and VLLM_URL == "http://localhost:8000":
        status = subprocess.run(
            ["docker", "inspect", "-f", "{{.State.Running}}", VLLM_CONTAINER],
            capture_output=True, text=True,
        )
        if status.stdout.strip() == "false":
            logs = subprocess.run(
                ["docker", "logs", "--tail", "80", VLLM_CONTAINER],
                capture_output=True, text=True,
            )
            raise RuntimeError(f"vLLM container exited:\n{logs.stderr[-6000:]}")
    elapsed = STARTUP_TIMEOUT - int(deadline - time.monotonic())
    if elapsed - last_status >= 30:
        print(f"  Still loading... ({elapsed}s elapsed)")
        last_status = elapsed
    time.sleep(5)
    MODEL = server_model()

if MODEL is None:
    raise RuntimeError(
        f"No vLLM server became ready at {VLLM_URL} within {STARTUP_TIMEOUT}s. "
        f"Inspect logs with: docker logs {VLLM_CONTAINER}"
    )

print(f"Connected to {VLLM_URL} - served model: {MODEL}")
print(f"Model configuration source: {MODEL_SOURCE}")

Notebook CUDA: NVIDIA GB10 (121.7 GiB total)
GPU memory currently free: 5.0 GiB
Connected to http://localhost:8000 - served model: qwen3.5-9b
Model configuration source: Qwen/Qwen3.5-9B


> **Note:** The vLLM server might need 1 or 2 minutes to be ready.

## Your First Local LLM Request

vLLM exposes an OpenAI-compatible API, so we use the standard `openai` Python client.

In [3]:
from openai import OpenAI
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")

Qwen3 supports a *thinking mode* that generates chain-of-thought reasoning before answering. We disable it with `enable_thinking: False` to keep responses short and predictable.

In [4]:
start = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user",
               "content": "What is PagedAttention in one sentence?"}],
    max_tokens=80,
    temperature=0.7,
    top_p=0.8,
    extra_body={"top_k": 20,
                "chat_template_kwargs": {"enable_thinking": False}},
)
elapsed = time.time() - start

print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
print(resp.choices[0].message.content)
print(f"\nUsage: {resp.usage.prompt_tokens} prompt + "
      f"{resp.usage.completion_tokens} completion = {resp.usage.total_tokens} total")

Response (22.12s, 51 tokens):
PagedAttention is a memory management technique for large language models that divides the attention key and value memory into fixed-size blocks, allowing the model to efficiently handle sequences longer than the available GPU memory by storing only the active blocks and swapping inactive ones out.

Usage: 21 prompt + 51 completion = 72 total


## Exploring Model Behavior

Beyond simple chat, the vLLM API lets you inspect model internals and control generation. For example, **Logprobs:** allows you to see the model's confidence in each token it generates, plus the alternatives it considered

In [5]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The capital of France is"}],
    max_tokens=15,
    temperature=0.0,
    logprobs=True,
    top_logprobs=5,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(f"Response: {resp.choices[0].message.content.strip()}\n")
print("Token-by-token probabilities:\n")

for tok in resp.choices[0].logprobs.content[:8]:
    print(f"  Chosen: '{tok.token}'  (logprob {tok.logprob:.2f})")
    if tok.top_logprobs:
        for alt in tok.top_logprobs[:5]:
            pct = math.exp(alt.logprob) * 100
            bar = "\u2588" * min(20, max(1, int(pct / 5)))
            print(f"    {pct:5.1f}%  {bar}  '{alt.token}'")
    print()

Response: The capital of France is **Paris**.

Located in the north-central part

Token-by-token probabilities:

  Chosen: 'The'  (logprob -0.04)
     95.9%  ███████████████████  'The'
      3.3%  █  'Paris'
      0.4%  █  '**'
      0.2%  █  'Par'
      0.0%  █  'London'

  Chosen: ' capital'  (logprob -0.00)
    100.0%  ███████████████████  ' capital'
      0.0%  █  ' answer'
      0.0%  █  ' **'
      0.0%  █  ' current'
      0.0%  █  ' sentence'

  Chosen: ' of'  (logprob -0.00)
    100.0%  ███████████████████  ' of'
      0.0%  █  ' city'
      0.0%  █  ' and'
      0.0%  █  ' ('
      0.0%  █  ' is'

  Chosen: ' France'  (logprob -0.00)
    100.0%  ███████████████████  ' France'
      0.0%  █  ' **'
      0.0%  █  ' the'
      0.0%  █  'France'
      0.0%  █  ' Paris'

  Chosen: ' is'  (logprob -0.00)
    100.0%  ███████████████████  ' is'
      0.0%  █  ','
      0.0%  █  ' has'
      0.0%  █  ' was'
      0.0%  █  ' ('

  Chosen: ' **'  (logprob -0.00)
    100.0%  ████████████

## Observing vLLM Under the Hood

vLLM exposes a Prometheus-compatible **`/metrics`** endpoint (it's a format that is easy to scrape). Key metrics to watch:

- **`num_requests_running / waiting`**: how many requests are active vs queued
- **`gpu_cache_usage_perc`** (or `cpu_cache_usage_perc`): KV cache memory pressure
- **`prompt_tokens_total / generation_tokens_total`**: cumulative token counts


In [6]:
def get_vllm_metrics(base_url=VLLM_URL):
    """Scrape vLLM Prometheus /metrics and return {name: value}."""
    r = requests.get(f"{base_url}/metrics")
    metrics = {}
    for line in r.text.split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        name = line.split("{")[0].split()[0]
        try:
            metrics[name] = float(line.split()[-1])
        except (ValueError, IndexError):
            continue
    return metrics

metrics = get_vllm_metrics()
print("Current vLLM Metrics:")
for key in ["vllm:num_requests_running", "vllm:num_requests_waiting",
            "vllm:gpu_cache_usage_perc", "vllm:cpu_cache_usage_perc",
            "vllm:prompt_tokens_total", "vllm:generation_tokens_total"]:
    if key in metrics:
        print(f"  {key.replace('vllm:', '')}: {metrics[key]:g}")

with open("outputs/metrics_snapshot.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nFull metrics saved to outputs/metrics_snapshot.json")

Current vLLM Metrics:
  num_requests_running: 0
  num_requests_waiting: 0
  prompt_tokens_total: 38
  generation_tokens_total: 66

Full metrics saved to outputs/metrics_snapshot.json


## Continuous Batching in Action

vLLM uses **continuous batching** (iteration-level scheduling): when a request finishes generating, its slot is immediately filled by the next waiting request.

Let's send 5 requests **concurrently** and watch the metrics while they run.

In [7]:
import concurrent.futures

prompts = [
    "What is quantization?",
    "Explain KV caching briefly.",
    "What is continuous batching?",
    "Why is LLM inference memory-bound?",
    "What is PagedAttention?",
]

def _ask(prompt):
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

before = get_vllm_metrics()
print(f"Sending {len(prompts)} concurrent requests...\n")
start = time.time()
peak_running = 0
peak_waiting = 0

with concurrent.futures.ThreadPoolExecutor(
    max_workers=len(prompts)) as pool:
    futures = {pool.submit(_ask, prompt): prompt for prompt in prompts}

    while not all(future.done() for future in futures):
        sample = get_vllm_metrics()
        peak_running = max(
            peak_running, int(sample.get("vllm:num_requests_running", 0))
        )
        peak_waiting = max(
            peak_waiting, int(sample.get("vllm:num_requests_waiting", 0))
        )
        time.sleep(0.1)

    for future in concurrent.futures.as_completed(futures):
        response = future.result()
        prompt = futures[future]
        print(f"  done: \"{prompt[:40]}\" -> {response.usage.completion_tokens} tokens")

elapsed = time.time() - start
after = get_vllm_metrics()
tokens = after.get("vllm:generation_tokens_total", 0) - before.get(
    "vllm:generation_tokens_total", 0
)

print(f"\nPeak running: {peak_running}  |  peak waiting: {peak_waiting}")
print(f"All {len(prompts)} completed in {elapsed:.2f}s")
if tokens > 0:
    print(f"Tokens generated: {tokens:g}  |  ~{tokens / elapsed:.1f} tokens/s")

Sending 5 concurrent requests...

  done: "What is continuous batching?" -> 120 tokens
  done: "Why is LLM inference memory-bound?" -> 120 tokens
  done: "What is quantization?" -> 120 tokens
  done: "What is PagedAttention?" -> 120 tokens
  done: "Explain KV caching briefly." -> 120 tokens

Peak running: 5  |  peak waiting: 0
All 5 completed in 9.93s
Tokens generated: 600  |  ~60.4 tokens/s


> **Reading the result:** the cell polls `/metrics` until all requests finish and reports the observed peak instead of relying on one race-prone snapshot. A peak below five is still possible when short requests complete before all requests become active; increase `max_tokens` or the request count to create sustained GPU pressure.

**What Just Happened**

vLLM's scheduler received all 5 requests and processed them using **continuous batching**. Each generation step produces tokens for every request in the batch, and completed requests free their slots immediately.

vLLM's **PagedAttention** is key to making this work at scale: the KV cache is divided into fixed-size blocks that can be placed anywhere in memory. When a request finishes, its blocks are immediately available for reuse — no fragmentation.

## Prefix Caching

Many applications share a long **system prompt** across requests. Without prefix caching, vLLM recomputes the KV cache for the shared prefix every time.

With **prefix caching**, vLLM detects shared prefixes and **reuses cached KV entries**. The first request pays the full prefill cost; subsequent requests skip it.

Let's send 5 requests with the same system prompt and compare timing.

In [8]:
SYSTEM_PROMPT = (
    "You are a technical teaching assistant for GPU-based LLM inference. "
    "Answer precisely and distinguish model weights, activations, and KV cache. "
    "Explain assumptions before drawing performance conclusions. "
    "Use no more than two sentences. "
    * 8
)

questions = [
    "What is weight quantization?",
    "How does vLLM handle memory?",
    "What is continuous batching?",
    "Why use prefix caching?",
    "What is GPTQ?",
]

before = get_vllm_metrics()
timings = []

print("Sending 5 requests with the SAME long system prompt...\n")
for index, question in enumerate(questions, start=1):
    started = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    elapsed = time.time() - started
    timings.append(elapsed)
    tokens = response.usage.completion_tokens
    ms_per_token = (elapsed / tokens * 1000) if tokens else 0.0
    print(
        f"  [{index}] {question:<35} {elapsed:.2f}s "
        f"({tokens} tok, {ms_per_token:.0f} ms/tok)"
    )

after = get_vllm_metrics()
query_key = "vllm:prefix_cache_queries_total"
hit_keys = [key for key in after if "prefix" in key.lower() and "hit" in key.lower()]
query_delta = after.get(query_key, 0) - before.get(query_key, 0)

print(f"\nPrefix-cache queries observed: {query_delta:g}")
if hit_keys:
    for key in sorted(hit_keys):
        hit_delta = after.get(key, 0) - before.get(key, 0)
        print(f"Prefix-cache hits ({key}): {hit_delta:g}")
else:
    print("No prefix-cache hit counter is exposed by this vLLM version.")

print(f"First request: {timings[0]:.2f}s")
print(f"Median repeated request: {sorted(timings[1:])[len(timings[1:]) // 2]:.2f}s")

Sending 5 requests with the SAME long system prompt...

  [1] What is weight quantization?        5.38s (60 tok, 90 ms/tok)
  [2] How does vLLM handle memory?        5.38s (60 tok, 90 ms/tok)
  [3] What is continuous batching?        5.39s (60 tok, 90 ms/tok)
  [4] Why use prefix caching?             5.24s (59 tok, 89 ms/tok)
  [5] What is GPTQ?                       5.44s (60 tok, 91 ms/tok)

Prefix-cache queries observed: 1715
Prefix-cache hits (vllm:external_prefix_cache_hits_created): 0
Prefix-cache hits (vllm:external_prefix_cache_hits_total): 0
Prefix-cache hits (vllm:prefix_cache_hits_created): 0
Prefix-cache hits (vllm:prefix_cache_hits_total): 0
First request: 5.38s
Median repeated request: 5.39s


**Why Prefix Caching Matters**

A positive cache-hit delta is direct evidence that vLLM reused KV blocks for the shared prefix. An increase in cache queries alone only proves that vLLM checked the cache. Timing is supporting evidence, but GPU scheduling and generation length also affect it. Long, repeated prefixes make the benefit easier to observe than short prompts.

---

## (Optional) KV Cache Size Visualization for the Served Model

The KV cache stores keys and values for **full-attention layers** and grows linearly with sequence length and concurrency. Qwen3.5 also has linear-attention layers with recurrent state; that fixed per-request state is not included in this simplified KV estimate.

In [9]:
from transformers import AutoConfig

model_config = AutoConfig.from_pretrained(MODEL_SOURCE)
text_config = getattr(model_config, "text_config", model_config)
layer_types = getattr(
    text_config,
    "layer_types",
    ["full_attention"] * text_config.num_hidden_layers,
)
full_attention_layers = sum(
    layer_type == "full_attention" for layer_type in layer_types
)
num_kv_heads = text_config.num_key_value_heads
head_dim = getattr(
    text_config,
    "head_dim",
    text_config.hidden_size // text_config.num_attention_heads,
)
dtype_bytes = 2  # BF16 KV cache; use 1 when the server uses an FP8 KV cache.
per_token = (
    2 * full_attention_layers * num_kv_heads * head_dim * dtype_bytes
)

print(f"Full-attention KV cache estimate - {MODEL_SOURCE}")
print(
    f"Layers: {full_attention_layers} full attention / "
    f"{text_config.num_hidden_layers} total"
)
print(
    f"Per token: 2 x {full_attention_layers} x {num_kv_heads} x "
    f"{head_dim} x {dtype_bytes} = {per_token:,} bytes "
    f"({per_token / 1024:.1f} KiB)\n"
)
print(f"  {'Context':>8}  {'KV cache/request':>18}")
print(f"  {'-' * 8}  {'-' * 18}")
for context_length in [1, 64, 256, 1024, 4096, 8192]:
    size = per_token * context_length
    if size < 1024**2:
        label = f"{size / 1024:.1f} KiB"
    else:
        label = f"{size / 1024**2:.1f} MiB"
    print(f"  {context_length:>7}t  {label:>18}")

concurrency = 10
context_length = 8192
total_gib = per_token * context_length * concurrency / 1024**3
print(f"\n{concurrency} concurrent x {context_length} tokens = {total_gib:.2f} GiB")
print("Qwen3.5 linear-attention recurrent state is additional.")

Full-attention KV cache estimate - Qwen/Qwen3.5-9B
Layers: 8 full attention / 32 total
Per token: 2 x 8 x 4 x 256 x 2 = 32,768 bytes (32.0 KiB)

   Context    KV cache/request
  --------  ------------------
        1t            32.0 KiB
       64t             2.0 MiB
      256t             8.0 MiB
     1024t            32.0 MiB
     4096t           128.0 MiB
     8192t           256.0 MiB

10 concurrent x 8192 tokens = 2.50 GiB
Qwen3.5 linear-attention recurrent state is additional.


---

## (Optional) Thinking Mode

Qwen3 supports a *thinking mode* where the model generates internal chain-of-thought reasoning (`<think>...</think>`) before the visible answer. This produces better answers but uses **significantly more tokens** — more KV cache, more compute, longer response.

Let's stream both modes side by side on the same prompt.

In [10]:
prompt = "What makes continuous batching better than static batching?"

for label, thinking, max_tok in [
    ("Thinking OFF", False, 80), ("Thinking ON", True, 200)]:
    print(f"=== {label} ===\n")
    start = time.time()
    tokens = 0
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tok, temperature=0.7, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": thinking}},
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            sys.stdout.write(chunk.choices[0].delta.content)
            sys.stdout.flush()
            tokens += 1
    elapsed = time.time() - start
    print(f"\n  [{tokens} tokens, {elapsed:.2f}s]\n")

=== Thinking OFF ===

Continuous batching is generally considered superior to static batching in modern distributed computing environments, particularly for large-scale machine learning training and data processing. Its primary advantage lies in its ability to **maximize hardware utilization and minimize idle time**, which directly translates to faster completion times and lower costs.

Here is a detailed breakdown of what makes continuous batching better:

### 1. Elimination of Idle Time (The
  [80 tokens, 7.03s]

=== Thinking ON ===


  [0 tokens, 17.68s]



## Summary

In this notebook, you:

- Served the newer **Qwen3.5-9B** model with CUDA-backed vLLM in text-only mode
- Kept the compression lab's **Qwen3-8B GPTQ** checkpoint as a lower-memory comparison
- Connected through the OpenAI-compatible API and inspected token log probabilities
- Used the **`/metrics` endpoint** to observe GPU cache usage and request counts
- Measured continuous batching with repeated metric sampling
- Distinguished prefix-cache queries from confirmed cache hits
- Estimated full-attention KV-cache memory from the served model's configuration
- Compared thinking and non-thinking modes

## Resources

- [Qwen3.5-9B model card](https://huggingface.co/Qwen/Qwen3.5-9B)
- [vLLM Qwen3.5 and Qwen3.6 usage guide](https://docs.vllm.ai/projects/recipes/en/latest/Qwen/Qwen3.5.html)
- [vLLM documentation](https://docs.vllm.ai/en/latest/)